In [1]:
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException,ElementNotInteractableException, NoSuchElementException

#from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(10) # 隱性等待
driver.get("https://today.line.me/tw/v2/movie/incinemas/playing")

# 初始化 ActionChains
action = ActionChains(driver)


### 利用 While Loop 與判斷式實作自動化網頁無限滾動

本小節將展示如何透過無限迴圈 (`while True`) 搭配條件判斷式 (`if`)，來應對現代網頁常見的**懶加載 (Lazy Loading)** 機制。透過模擬人類向下滾動網頁的行為，我們可以持續觸發 AJAX 請求，直到抓取完整頁面資料。

**核心執行步驟：**
1. **狀態追蹤**：設定 `last_count` 變數，用於紀錄前一次滾動後的資料總筆數。
2. **終止條件判斷**：在迴圈內抓取當前頁面的元素總數 (`current_count`)。若兩者數量相等 (`current_count == last_count`)，即代表網頁已抵達底部，無新資料載入，此時執行 `break` 終止迴圈。
3. **動態滾動觸發**：若尚有新資料，則定位出當前清單中的「最後一個元素」，並運用 `ActionChains` 將視窗滾動至該元素位置，迫使網頁載入後續資料。

In [2]:
# 初始化前一次抓取的電影數量
last_count = 0

# 持續滾動葉面，直到沒有新資料載入為止
while True:
    # 1. 取得目前頁面上的所有電影元素
    movie_list_whole = driver.find_elements(By.CSS_SELECTOR, 'div.detailListItem.movieListing-movie')
    current_count = len(movie_list_whole)

    # 第一個迴圈 current_count: last_count:
    # 第二個迴圈 current_count: last_count:
    # 第三個迴圈 current_count: last_count:
    # 第四個迴圈 current_count: last_count:

    # 判斷新資料是否有載入
    if current_count == last_count:
        print("數量未增加，已抵達底部")
        break

    # 找出目前清單的「最後一個元素」
    last_element = movie_list_whole[-1]

    # 使用 ActionChain 滾動到最後一個元素
    # 這會讓瀏覽器將視窗滾動到該元素出現的位置，會觸發 AJAX (Lazyloading)
    action.scroll_to_element(last_element).perform()

    # 等待資料載入
    time.sleep(2)

    # 更新 last_count
    last_count = current_count

    # 第一個迴圈 current_count: last_count:
    # 第二個迴圈 current_count: last_count:
    # 第三個迴圈 current_count: last_count:

數量未增加，已抵達底部


### 利用enumerate 將以取得的電影資訊篩選出我們要的內容存入清單內


#### 一、 核心教學目標：利用 `enumerate` 篩選並儲存資料
在處理多筆資料時，**「利用 `enumerate` 將已取得的電影資訊篩選出我們要的內容存入清單內」** 是一個非常標準且優雅的設計模式。`enumerate` 不僅能讓我們遍歷元素集合（`movie_list_whole`），還能同時自動生成索引值（`i`），這對於追蹤爬取進度、記錄日誌（Log）或在發生例外錯誤時進行除錯都極具價值。

#### 二、 Selenium 關鍵取值方法
在上述的迴圈中，我們主要練習兩種最常用的 Selenium 資料萃取屬性與方法：

1.  **`.text` 屬性**：
    * **用途**：取得兩個 HTML 標籤之間包含的純文字內容。
    * **應用場景**：當我們需要標題、評分、分級等直接顯示在網頁上的可見文字時使用。例如從 `<h2>標題</h2>` 中提取「標題」。
2.  **`.get_attribute('屬性名稱')` 方法**：
    * **用途**：取得 HTML 標籤內的特定屬性值。
    * **應用場景**：網頁中有許多重要資訊不會直接顯示為文字，而是隱藏在標籤屬性中。例如超連結的網址（`href`）、圖片的來源（`src`），或是表單元素的名稱（`name`）。在此範例中，我們利用 `.get_attribute('href')` 來精準提取時刻表的目標網址。


In [ ]:
# 建立一個空的容器，放置電影資訊
movie_list = []

# 遍歷所有電影元素並提取詳細的電影資訊
for i, element in enumerate(movie_list_whole):
    try:
        # 取得電影標題
        '''
        CSS Selector:
        h2.detailListItem-title.header.header--sm.header--primary.header--ellipsis1
        '''
        title = element.find_element(By.CSS_SELECTOR, 'h2.detailListItem-title.header.header--sm.header--primary.header--ellipsis1').text

        # 取得評分
        '''
        CSS Selector :
        span.iconInfo-text.text.text--f.text--secondary.text--regular
        '''
        score_CSS = 'span.iconInfo-text.text.text--f.text--secondary.text--regular'
        score = element.find_element(By.CSS_SELECTOR, score_CSS).text

        # 取得分級 
        '''
        CSS Selector :
        span.glnBadge-text.text.text--fNarrow.text--secondary.text--regular
        '''
        # 初始化分級元素
        content_rating = "無分級資訊"
        CSS_content_rating = 'span.glnBadge-text.text.text--fNarrow.text--secondary.text--regular'
        content_rating =element.find_element(By.CSS_SELECTOR,CSS_content_rating).text

        # 取得時刻表連結
        #利用 .get_attribute('href') 來精準提取時刻表的目標網址
        '''
        CSS Selector:
        a.detailListItem-bookingButton
        '''
        showtime_element = element.find_element(By.CSS_SELECTOR, 'a.detailListItem-bookingButton')
        showtime_url = showtime_element.get_attribute('href')

    # 若找不到元素，程式不會報錯，而是執行此區塊
    # 可以選擇裡面填寫 pass 或紀錄log
    except NoSuchElementException:
        pass

    print(f"{i+1}, {title}, 評分:{score}, 分級:{content_rating}, 場次連結: {showtime_url}")

    # 將單一電影的所有屬性打包為字典 (Dictionary)，並附加至 movie_list 串列的最尾端
    # 建立鍵值對 (Key-Value Pair) 以對應萃取出的變數
    movie_list.append({
        "title" : title,
        "Score" : score,
        "Content Rating": content_rating,
        "showtime_url" : showtime_url,
    })
    
        
        

1, 超級瑪利歐銀河電影版, 評分:8.6, 分級:普遍級, 場次連結: https://today.line.me/tw/v3/movie/BEQD020/1
2, 狸想世界, 評分:9.2, 分級:普遍級, 場次連結: https://today.line.me/tw/v3/movie/vXVJwB0/1
3, 李克寧 墓乃伊, 評分:9.0, 分級:限制級, 場次連結: https://today.line.me/tw/v3/movie/0MNL20p/1
4, 死亡之握：鬼牽手, 評分:8.1, 分級:輔15級, 場次連結: https://today.line.me/tw/v3/movie/RB8P27j/1
5, 極限返航, 評分:9.1, 分級:保護級, 場次連結: https://today.line.me/tw/v3/movie/9mll09x/1
6, 抓馬戀人, 評分:8.5, 分級:輔15級, 場次連結: https://today.line.me/tw/v3/movie/1DyxPWp/1
7, 劇場版 銀河特急 Milky☆Subway, 評分:9.5, 分級:保護級, 場次連結: https://today.line.me/tw/v3/movie/eL5WGyD/1
8, No Good！歐吉桑, 評分:8.4, 分級:普遍級, 場次連結: https://today.line.me/tw/v3/movie/vXM6jD3/1
9, 海街日記, 評分:9.8, 分級:普遍級, 場次連結: https://today.line.me/tw/v3/movie/EXeDZqn/1
10, 劇場版「暗殺教室」: 我們的時光, 評分:8.3, 分級:保護級, 場次連結: https://today.line.me/tw/v3/movie/rmQ79vn/1
11, 陽光女子合唱團, 評分:9.0, 分級:輔12級, 場次連結: https://today.line.me/tw/v3/movie/1Dnwjm3/1
12, 高雄有顆藍寶石, 評分:9.5, 分級:保護級, 場次連結: https://today.line.me/tw/v3/movie/7Nqmz3M/1
13, 弒婚遊戲：2度開局, 評分:8.6, 分級:限制級, 場次連結: ht

In [ ]:
'''
等等會用到
'''
FULL_STAR_PATH = 'M12 18.344l-5.81 3.609c-.147.091-.34.046-.43-.1-.043-.07-.057-.152-.04-.23l1.469-6.96-5.09-4.736c-.126-.118-.133-.316-.016-.442.052-.056.122-.091.198-.099l6.746-.68 2.684-6.513c.066-.16.249-.235.408-.17.077.032.138.094.17.17l2.684 6.514 6.746.68c.172.017.297.17.28.342-.008.075-.043.146-.099.198l-5.09 4.736 1.47 6.96c.036.169-.072.334-.24.37-.08.017-.161.002-.23-.04L12 18.343z'
HALF_STAR_PATH = 'M12.12 2.024c.076.031.137.093.169.17l2.684 6.513 6.746.68c.172.017.297.17.28.342-.008.075-.043.146-.099.198l-5.09 4.736 1.47 6.96c.036.169-.072.334-.24.37-.08.017-.161.002-.23-.04L12 18.343l-5.81 3.61c-.147.091-.34.046-.43-.1-.043-.07-.057-.152-.04-.23l1.469-6.96-5.09-4.736c-.126-.118-.133-.316-.016-.442.052-.056.122-.091.198-.099l6.746-.68 2.684-6.513c.066-.16.249-.235.408-.17zM12 6.463v9.651l3.662 2.275-.925-4.383 3.316-3.086-4.398-.443L12 6.463z'

# 初始化一個主列表來存放評論資料
all_reviews_data =[]

for element in movie_list:
    # 取的單一電影的場次連結
    individual_link = element['showtime_url']

    # 取得電影名稱
    movie_name = element['title']

    print(f"正在爬取電影:{movie_name}...")

    # 前往該電影的詳細頁面
    driver.get(individual_link)

    try:
        '''
        一、挑戰(ㄧ): 訪問詳細頁面時，等待最多 10 秒，直到「網友短評」這個分頁元素可以點擊
            教學目標:
                - 設定並練習使用顯式等待 (Explicit Wait)的策略
            學員實作:
                - 建立一個等待物件，設定最長的等待時間為 10 秒
                - 後續所有的等待操作都須遵守這個超時設定
        '''
        wait = WebDriverWait(driver,10)

        '''
        二、挑戰(二): 點擊「網友短評」這個分頁元素
           教學目標:
                - 當有多個元素共用同一個 CSS選擇器字串，要等到這些元素均出現，且「網友短評」這個分頁元素可以點擊
            學員實作:
                - 將 CSS 選擇器字串存放到一個變數中，方便後續維護與修改
                註 : CSS_Selector : 'a.ltcp-link.tabLink.movie.css-1hq811z'
                - 撰寫邏輯使程式等待直到所有符合選擇器的分頁連結元素都出現在頁面中      
        '''

        # Selenium 的 expected_condition 模組中，所有與尋找元素相關的條件類別，其均設計為接收 「一個」參數
        # 而該參數被嚴格定義為一個包含兩個字串原物的元組(Tuple)
        # 因此當我們利用By策略時，傳入presence_of_all_elements_located這個方法的參數須為Tuple(By.CSS_SELECTOR, tab_selector)
        tab_selector = 'a.ltcp-link.tabLink.movie.css-1hq811z'
        elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, tab_selector)))
        
        '''
        「如果你問老師說 : 那如果 10 秒內都沒有等到呢?」
        :「那就會拋出 TimeoutException 異常，直行將跳轉到對應的 except 區塊」
        '''

        # 檢查分頁元素的數量
        if len(elements) < 3:
            print(f"電影{movie_name}的分頁數量不足，故跳過...")
            continue

        # 取得「網友短評」分頁元素
        review_tab_to_be_click = elements[-1]
 
        # 等到元素可以被點擊，後進行點擊
        wait.until(EC.element_to_be_clickable(review_tab_to_be_click)).click()

    # 捕捉在尋找或點擊「網友短評」元素過程中，最常發生的三種特定例外狀況：
    # 1. TimeoutException: 顯式等待（Explicit Wait）超過設定時間上限，目標元素仍未載入或不可見。
    # 2. IndexError: 嘗試透過索引值取出列表元素（例如 elements[0]）時，發現列表為空，意味著 DOM 樹中不存在符合選擇器的節點。
    # 3. ElementNotInteractableException: 目標元素雖然存在於 DOM 樹中，但可能被其他浮動元素（如廣告、彈出視窗）遮擋，或屬性為隱藏（Hidden），導致 WebDriver 無法執行點擊動作。
    except (TimeoutException, IndexError, ElementNotInteractableException) as e:
        print(f"電影:{movie_name}的網友短評無法互動，因為:{e}，故跳過")
        # 放棄當前這部電影的後續操作，直接中斷本次迭代（Iteration），並讓外層迴圈（For Loop）繼續執行下一部電影的爬取作業，避免程式崩潰。
        continue
    
    # 恭喜，如果你在這邊還活著的話，你已經可以順路進入到網友短評
    # 觀察每個「網友短評」，會發現依然要向下滾動網頁使瀏覽器載入新的評論

    # 初始化前一次抓取網友短評的數量
    last_count = 0
    
    while True:
        # 1. 取得目前頁面上所有的評論
        # CSS_Selector : div.css-hnvcda
        comment_list_whole = driver.find_elements(By.CSS_SELECTOR, 'div.css-hnvcda')
        current_count = len(comment_list_whole)

        # 2. 終止條件 : 判斷是否還有新資料載入
        # 如果目前數量等於上次數量，代表網頁已不再載入新內容，迴圈結束
        if current_count == last_count:
            break

        # 3. 鎖定目標 : 找出目前清單中的「最後一個」評論元素
        last_element = comment_list_whole[-1]

        # 4. 使用ActionChains 滾動到最後一個元素
        action.scroll_to_element(last_element).perform()

        # 5. 等待載入
        time.sleep(2)

        # 6.更新狀態 : 將目前的數量記錄下來，供下一輪比對
        last_count = current_count
    # 遍歷當前頁面的所有評論
    for comment_node in comment_list_whole:
        try:
            # 擷取星星數量
            # CSS_Selector : div[class=ratingStar] path
            star_paths = comment_node.find_elements(By.CSS_SELECTOR, 'div[class=ratingStar] path')

            # 初始化星星數量
            star = 0

            # 判定星星為整顆/半顆，並且計算星星數量
            for path in star_paths:
                # 取得 path 的 d 屬性值
                path_data = path.get_attribute('d')
                if path_data == FULL_STAR_PATH:
                    star += 1
                elif path_data == HALF_STAR_PATH:
                    star += 0.5

            # 擷取評論的內容文字賦值給comment_text
            # CSS_Selector : span.css-xcrf5c
            comment_text = comment_node.find_element(By.CSS_SELECTOR, 'span.css-xcrf5c').text

            #將擷取到的評論資料整理成字典，存入總清單中
            all_reviews_data.append({
                '電影名稱' : movie_name,
                '評分': star,
                '評論': comment_text,
            })
        # 用 
        except NoSuchElementException:
            print(f"電影{movie_name}的某則評論不完整，要跳過該評論")
            continue
# 將先前爬蟲迴圈中收集到包含多個字典(Dictionary)的串列(List) 'all_reviews_data'
# 作為引數傳入 pd.DataFrame() 建構子中。
# 此操作會將該串列轉換為二維的 Pandas DataFrame 物件，並將記憶體參考賦值給變數 'df'
df = pd.DataFrame(all_reviews_data)

driver.quit()
    

正在爬取電影:超級瑪利歐銀河電影版...
正在爬取電影:狸想世界...
正在爬取電影:李克寧 墓乃伊...
正在爬取電影:死亡之握：鬼牽手...
正在爬取電影:極限返航...
正在爬取電影:抓馬戀人...
正在爬取電影:劇場版 銀河特急 Milky☆Subway...
正在爬取電影:No Good！歐吉桑...
正在爬取電影:海街日記...
正在爬取電影:劇場版「暗殺教室」: 我們的時光...
正在爬取電影:陽光女子合唱團...
正在爬取電影:高雄有顆藍寶石...
正在爬取電影:弒婚遊戲：2度開局...
正在爬取電影:你罪乖...
正在爬取電影:如月車站2...
正在爬取電影:你是不會當樹嗎...
正在爬取電影:雙囍...
正在爬取電影:夏日大作戰(全新4K修復版)...
正在爬取電影:生命之詩(4K數位修復版)...
正在爬取電影:星與月是天上的洞...
正在爬取電影:咒死你...
正在爬取電影:尋秦記Plus+多元宇宙版...
正在爬取電影:捕風追影...
正在爬取電影:魅影巨星...
正在爬取電影:蠱降...
正在爬取電影:最後一槍...
正在爬取電影:A KITE 風箏...
正在爬取電影:3670...
正在爬取電影:邦尼殺死他...
正在爬取電影:人生海海...
正在爬取電影:63小時死亡對峙...
正在爬取電影:心機特務...
正在爬取電影:風流一代...
正在爬取電影:哈姆奈特...
正在爬取電影:港口天光...
正在爬取電影:重擊人生...
正在爬取電影:泰殺歌后戰...
正在爬取電影:猩瘋血雨...
正在爬取電影:少女與戰車 更加相親相愛作戰！第三幕...
正在爬取電影:深度安靜...
正在爬取電影:叫我驅魔男神...
正在爬取電影:28年毀滅倒數：人骨聖殿...
正在爬取電影:小勇士大冒險...
正在爬取電影:邪降：覺醒...
正在爬取電影:巧虎電影《勇氣之歌》...
正在爬取電影:舞林殺手...
正在爬取電影:自殺通告...
正在爬取電影:尋秦記...
正在爬取電影:關鍵公敵...
正在爬取電影:機動戰士鋼彈 閃光的哈薩威 喀耳刻的魔女...
正在爬取電影:秒速5公分(導演特別映像版)...
正在爬取電影:四人行，不行...
正在爬取電影:小雨愛蜜莉...
正在爬取電影:穿越地獄之門...
正在爬取電影:如月車站...
正在爬取電影:女孩不平

存成 csv 檔案

In [7]:
# 檢視資料表前 5 列 (Rows)，確認資料是否正確載入且行名 (Columns) 對齊無誤
print("資料表前 5 列:")
print(df.head())

資料表前 5 列:
         電影名稱   評分                                                 評論
0  超級瑪利歐銀河電影版  5.0                            好好看，耀西可愛破表，背景音樂和電動畫面都很棒
1  超級瑪利歐銀河電影版  5.0  首映當天沒什麼人，看杜比影廳。\n畫面很美，劇情還不錯。\n有一些別的電影梗（星際異攻隊）🦊...
2  超級瑪利歐銀河電影版  5.0                        真的太好看了！非常適合帶小朋友一起觀賞\n有趣刺激好玩
3  超級瑪利歐銀河電影版  4.0                           第一集劇情比較好\n第二集很多彩蛋\n期待第三集
4  超級瑪利歐銀河電影版  4.5                                            耀西🦖 好可愛


In [9]:
# 3. 檢視數值型資料行的敘述性統計 (如：評分的最大值、最小值、平均值)
print("\n數值資料敘述性統計:")
print(df.describe())


數值資料敘述性統計:
               評分
count  852.000000
mean     4.480634
std      1.090193
min      0.500000
25%      4.500000
50%      5.000000
75%      5.000000
max      5.000000


**`encoding='utf-8-sig'`**

：
**字元編碼參數**

。這是針對繁體中文語系至關重要的設定。一般寫入文字檔常使用

```
'utf-8'
```

，但在微軟的 Excel 軟體中直接開啟普通的 UTF-8 檔案時，經常會出現中文亂碼。加上

```
-sig
```

後，Pandas 會在檔案開頭寫入

**BOM（Byte Order Mark，位元組順序記號）**

，藉此強制作業系統與 Excel 識別該檔案為 UTF-8 編碼，確保中文字元得以正確渲染。

In [10]:
#  將清理或確認無誤的資料匯出為 CSV 檔案進行持久化儲存
# 參數說明：index=False 避免將 DataFrame 預設的整數索引寫入檔案；encoding='utf-8-sig' 確保繁體中文在 Excel 中不會出現亂碼
df.to_csv('movie_reviews.csv', index=False, encoding='utf-8-sig')